In [ ]:
import json
import re
from typing import List, Dict

# 视频和图片业务关键参数
VIDEO_KEY_PARAMS = ["model_name", "latency_sec", "video_fps", "video_resolution"]
IMAGE_KEY_PARAMS = ["model_name", "batch_size", "image_resolution"]

def parse_intent_output(llm_text: str) -> Dict:
    """
    从 LLM 输出中提取意图解析 JSON 并返回 state。
    支持 Markdown 包裹 ```json ...```，兼容缺失参数检测。
    """
    # 1. 尝试用正则提取 ```json ...``` 块
    json_str = ""
    match = re.search(r"```json(.*?)```", llm_text, re.DOTALL | re.IGNORECASE)
    if match:
        json_str = match.group(1).strip()
    else:
        # 如果没有 ```json``` 包裹，尝试取第一个 { ... } 块
        first_brace = llm_text.find("{")
        last_brace = llm_text.rfind("}")
        if first_brace != -1 and last_brace != -1:
            json_str = llm_text[first_brace:last_brace+1]

    # 2. 解析 JSON，容错
    try:
        intent_result = json.loads(json_str)
    except Exception as e:
        print("JSON解析失败:", e)
        intent_result = {"business_type": None, "parameters": {}}

    # 3. 检查关键参数是否缺失
    business_type = intent_result.get("business_type")
    params = intent_result.get("parameters", {})

    if business_type == "视频AI推理":
        key_params = VIDEO_KEY_PARAMS
    elif business_type == "图片批量推理":
        key_params = IMAGE_KEY_PARAMS
    else:
        # 未识别业务类型，视为缺失
        key_params = []

    missing_params = [k for k in key_params if not params.get(k)]
    stage = "ask_missing" if missing_params else "complete"
    parse_success = business_type is not None and not missing_params

    # 4. 构建 state 返回
    state = {
        "type": "state",
        "stage": stage,
        "workflow": "intent_parsing",
        "parse_success": parse_success,
        "missing_params": missing_params,
        "intent_result": intent_result
    }

    return state

    

In [ ]:
text='''意图解析结果状态：缺失参数  
缺失关键参数：model_name  
```json
{
  "type": "state",
  "stage": "ask_missing",
  "workflow": "intent_parsing",
  "parse_success": false,
  "missing_params": ["model_name"],
  "intent_result": {
    "business_type": "视频AI推理",
    "parameters": {
      "model_name": null,
      "latency_sec": 2,
      "video_fps": 25,
      "video_resolution": "1920x1080",
      "modality": "低时延转发模态"
    }
  }
}
```
请补充缺失的关键参数：model_name（例如使用的模型名称，如 yolov8、ssd 等）。'''
state = parse_intent_output(text)
print(json.dumps(state, indent=2, ensure_ascii=False))


In [ ]:
from server.parser.state import State
s=State()
json.dumps(s.model_dump())

In [20]:
import dateparser
import re

def zh_num_to_arabic(text: str) -> str:
    # 简单映射中文数字 0-9
    mapping = {"一":"1", "二":"2", "三":"3", "四":"4", "五":"5", "六":"6", "七":"7", "八":"8", "九":"9", "十":"10"}
    for k, v in mapping.items():
        text = text.replace(k, v)
    return text

user_input = "一小时后"
user_input = zh_num_to_arabic(user_input)  # -> "1小时后"

dt = dateparser.parse(
    user_input,
    languages=['zh'],
    settings={'TIMEZONE': 'Asia/Shanghai', 'RETURN_AS_TIMEZONE_AWARE': False}
)
if not dt:
    print("解析失败，请输入 'YYYY-MM-DD HH:MM'")
else:
    print("解析成功:", dt)

解析成功: 2026-04-07 17:45:36.916402


In [21]:
print("ask_missing" if [] or [] else "complete")

complete
